# 21 - TransMorph-style en todos los sujetos

Este notebook resume las tres configuraciones TransMorph-style probadas sobre los seis sujetos y compara la configuracion final con el baseline clasico y VoxelMorph-style.

## Preguntas que responde

1. Cuanto mejora cada configuracion respecto al baseline clasico.
2. Si TransMorph-style supera al VoxelMorph-style seleccionado.
3. Cuanto desplazamiento y plegamiento introduce la deformacion.
4. Que configuracion ofrece el compromiso mas seguro.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

ROOT = Path.cwd()
METHOD_DIR = ROOT / 'Registration' / 'DeepLearning' / 'Tecnicas_registration' / '02_transmorph'
GRID_DIR = METHOD_DIR / 'outputs' / 'outputs_transmorph_grid'
FINAL_DIR = METHOD_DIR / 'outputs' / 'outputs_transmorph_final_block'
GRID_SCRIPT = METHOD_DIR / 'scripts' / 'run_transmorph_grid.py'

CONFIG_CSV = GRID_DIR / 'transmorph_grid_config_summary.csv'
GRID_CSV = GRID_DIR / 'transmorph_grid_summary.csv'
FINAL_CSV = FINAL_DIR / 'transmorph_final_selected_summary.csv'

print('Grid:', GRID_DIR)
print('Final:', FINAL_DIR)

In [ ]:
# Cambia a True solo si quieres repetir las 18 optimizaciones.
# El script reutiliza resultados existentes cuando encuentra sus JSON.
RUN_FULL_GRID = False

if RUN_FULL_GRID:
    result = subprocess.run([sys.executable, str(GRID_SCRIPT)], cwd=str(ROOT), text=True, capture_output=True)
    print(result.stdout[-8000:])
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('Fallo la grid completa TransMorph-style')

In [ ]:
config_df = pd.read_csv(CONFIG_CSV)
display(config_df.round(4))

In [ ]:
selected_df = pd.read_csv(FINAL_CSV)
columns = [
    'subject', 'classical_baseline_dice', 'voxelmorph_selected_dice',
    'transmorph_dice', 'transmorph_minus_baseline',
    'transmorph_minus_voxelmorph', 'flow_p95_px',
    'negative_jacobian_fraction', 'quality_status',
]
display(selected_df[columns].round(4))

means = pd.Series({
    'Classical baseline': selected_df['classical_baseline_dice'].mean(),
    'VoxelMorph selected': selected_df['voxelmorph_selected_dice'].mean(),
    'TransMorph-style selected': selected_df['transmorph_dice'].mean(),
}, name='mean_dice')
display(means.to_frame().round(4))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 13))
axes[0].imshow(Image.open(GRID_DIR / 'transmorph_regularization_tradeoff.png'))
axes[0].axis('off')
axes[0].set_title('Trade-off de regularizacion')

axes[1].imshow(Image.open(FINAL_DIR / 'transmorph_selected_method_comparison.png'))
axes[1].axis('off')
axes[1].set_title('Baseline clasico vs VoxelMorph vs TransMorph-style')
plt.tight_layout()

In [ ]:
plt.figure(figsize=(18, 14))
plt.imshow(Image.open(FINAL_DIR / 'transmorph_selected_summary_montage.png'))
plt.axis('off')
plt.title('Resumen visual de la configuracion TransMorph-style seleccionada')
plt.tight_layout()

## Lectura critica

- La configuracion flexible `tm_s040_d012` alcanza un Dice medio cercano al VoxelMorph seleccionado, pero conserva un caso rechazable por deformacion.
- La configuracion conservadora `tm_s100_d008` reduce desplazamientos y evita casos clasificados como rechazo final, aunque pierde Dice.
- SB012 y SB013 son los casos mas favorables.
- SB017 mejora mucho respecto al baseline, pero sigue necesitando revision por deformacion.
- SB018 y SB019 no mejoran al baseline con la configuracion conservadora.

**Conclusion:** TransMorph-style constituye una segunda familia deep learning valida como exploracion, pero VoxelMorph-style sigue siendo el resultado deep principal para este dataset.